In [1]:
# motor_learning_refactored.py

import os
os.environ['OMP_NUM_THREADS'] = '1'

from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
import json
import pickle
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, mannwhitneyu, gaussian_kde
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import pingouin as pg
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION
# ==============================================================================

@dataclass
class Config:
    """Analysis configuration."""
    base_output_dir: Path = Path('motor_learning_output')
    min_complete_strides: int = 20
    motor_noise_threshold: float = 0.3
    figure_dpi: int = 300
    alpha_level: float = 0.05
    
    trial_types: List[str] = field(default_factory=lambda: ['vis1', 'invis', 'vis2'])
    conditions: List[str] = field(default_factory=lambda: ['max', 'min'])
    
    def __post_init__(self):
        """Create directory structure."""
        self.figures_dir = self.base_output_dir / 'figures'
        self.reports_dir = self.base_output_dir / 'reports'
        self.data_dir = self.base_output_dir / 'data'
        
        for directory in [self.figures_dir, self.reports_dir, self.data_dir]:
            directory.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# DATA STRUCTURES
# ==============================================================================

@dataclass
class Subject:
    """Individual subject data."""
    subject_id: str
    age: float
    metadata: Dict[str, Any]
    trial_data: Dict[str, pd.DataFrame]  # trial_type -> dataframe
    
    def get_trial(self, trial_type: str) -> Optional[pd.DataFrame]:
        """Get trial data if it exists."""
        return self.trial_data.get(trial_type)

# ==============================================================================
# DATA LOADING
# ==============================================================================

class DataLoader:
    """Load and organize motor learning data."""
    
    def __init__(self, metadata_path: str, data_root_dir: str):
        self.metadata = pd.read_csv(metadata_path)
        self.data_root = Path(data_root_dir)
        self.subjects = {}
        
    def load_all_subjects(self) -> Dict[str, Subject]:
        """Load data for all subjects."""
        for _, row in self.metadata.iterrows():
            subject = self._load_subject(row)
            if subject:
                self.subjects[subject.subject_id] = subject
        return self.subjects
    
    def _load_subject(self, metadata_row: pd.Series) -> Optional[Subject]:
        """Load data for a single subject."""
        subject_id = metadata_row['ID']
        subject_dir = self.data_root / subject_id
        
        if not subject_dir.exists():
            return None
        
        # Load trial data
        trial_data = {}
        trial_mappings = {
            'primer': 'vis1',
            'trial': 'invis', 
            'vis': 'vis2',
            'pref': 'pref'
        }
        
        for file_prefix, trial_type in trial_mappings.items():
            df = self._load_trial_files(subject_dir, file_prefix)
            if df is not None:
                trial_data[trial_type] = self._process_trial_data(df)
        
        if not trial_data:
            return None
            
        return Subject(
            subject_id=subject_id,
            age=metadata_row.get('age_months', 0) / 12,
            metadata=metadata_row.to_dict(),
            trial_data=trial_data
        )
    
    def _load_trial_files(self, subject_dir: Path, prefix: str) -> Optional[pd.DataFrame]:
        """Load and combine trial files."""
        files = sorted(subject_dir.glob(f"{prefix}*.txt"))
        if not files:
            return None
            
        # For preference trials, use largest file
        if prefix == 'pref' and len(files) > 1:
            files = [max(files, key=lambda f: f.stat().st_size)]
        
        # Load and combine files
        dfs = []
        for f in files:
            try:
                df = pd.read_csv(f, sep='\t')
                if not df.empty:
                    dfs.append(df)
            except:
                continue
        
        if not dfs:
            return None
            
        combined = pd.concat(dfs, ignore_index=True)
        
        # Clean up
        if 'Stride Number' in combined.columns:
            combined = combined.sort_values('Stride Number')
            combined = combined.drop_duplicates('Stride Number')
        
        return combined
    
    def _process_trial_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Process trial data."""
        if df is None or df.empty:
            return df
        
        # Add derived columns
        if all(col in df.columns for col in ['Upper bound success', 'Lower bound success']):
            df['Target size'] = df['Upper bound success'] - df['Lower bound success']
        
        # Scale sum of gains and steps
        if 'Sum of gains and steps' in df.columns:
            df['Sum of gains and steps'] = 1.5 * df['Sum of gains and steps']
        
        return df

# ==============================================================================
# METRICS CALCULATION
# ==============================================================================

class MetricsCalculator:
    """Calculate performance metrics."""
    
    @staticmethod
    def calculate_success_rate(df: pd.DataFrame, condition: str = None) -> float:
        """Calculate success rate for a trial or condition."""
        if df is None or 'Success' not in df.columns:
            return np.nan
        
        if condition:
            df = MetricsCalculator._filter_condition(df, condition)
        
        if df.empty:
            return np.nan
            
        return df['Success'].mean()
    
    @staticmethod
    def calculate_mean_stride_length(df: pd.DataFrame, condition: str = None) -> float:
        """Calculate mean stride length."""
        if df is None or 'Sum of gains and steps' not in df.columns:
            return np.nan
        
        if condition:
            df = MetricsCalculator._filter_condition(df, condition)
        
        if df.empty:
            return np.nan
            
        return df['Sum of gains and steps'].mean()
    
    @staticmethod
    def calculate_stride_variability(df: pd.DataFrame, condition: str = None) -> float:
        """Calculate stride length variability (SD)."""
        if df is None or 'Sum of gains and steps' not in df.columns:
            return np.nan
        
        if condition:
            df = MetricsCalculator._filter_condition(df, condition)
        
        if df.empty:
            return np.nan
            
        return df['Sum of gains and steps'].std()
    
    @staticmethod
    def calculate_learning(df: pd.DataFrame, condition: str = None) -> float:
        """Calculate learning metric: (avg_stride - preferred) / (target - preferred)."""
        if df is None or not all(col in df.columns for col in ['Sum of gains and steps', 'Constant']):
            return np.nan
        
        if condition:
            df = MetricsCalculator._filter_condition(df, condition)
        
        if df.empty:
            return np.nan
        
        avg_stride = df['Sum of gains and steps'].mean()
        target = df['Constant'].iloc[0] if not df['Constant'].empty else np.nan
        preferred = 2.0  # Normalized preferred stride
        
        if pd.isna(target) or abs(target - preferred) < 1e-6:
            return np.nan
            
        return (avg_stride - preferred) / (target - preferred)
    
    @staticmethod
    def calculate_retention(df: pd.DataFrame, clamp_num: int = 1) -> float:
        """Calculate retention during success clamp periods."""
        if df is None or df.empty:
            return np.nan
        
        # Find success clamp periods
        clamps = MetricsCalculator._find_success_clamps(df)
        
        if len(clamps) < clamp_num:
            return np.nan
        
        clamp = clamps[clamp_num - 1]
        
        # Get pre-clamp and clamp performance
        pre_clamp_perf = MetricsCalculator._get_pre_clamp_performance(df, clamp['start'])
        clamp_perf = clamp['data']['Sum of gains and steps'].tail(20).mean()
        
        if pre_clamp_perf is None:
            return np.nan
        
        baseline = 2.0
        learning_before = pre_clamp_perf - baseline
        learning_during = clamp_perf - baseline
        
        if abs(learning_before) < 0.01:
            return np.nan
            
        return (learning_during / learning_before) * 100
    
    @staticmethod
    def _filter_condition(df: pd.DataFrame, condition: str) -> pd.DataFrame:
        """Filter dataframe for specific condition (max/min target)."""
        if 'Target size' not in df.columns or 'Constant' not in df.columns:
            return df
        
        min_target = df['Target size'].min()
        min_target_periods = df[df['Target size'] <= min_target + 0.001]
        
        if min_target_periods.empty:
            return pd.DataFrame()
        
        if condition == 'max':
            const_value = min_target_periods['Constant'].max()
        else:
            const_value = min_target_periods['Constant'].min()
        
        period_data = min_target_periods[
            np.isclose(min_target_periods['Constant'], const_value, rtol=1e-5)
        ]
        
        return period_data.tail(20)  # Last 20 strides of condition
    
    @staticmethod
    def _find_success_clamps(df: pd.DataFrame) -> List[Dict]:
        """Find success clamp periods in trial."""
        if not all(col in df.columns for col in ['Target size', 'Constant', 'Stride Number']):
            return []
        
        max_target = df['Target size'].max()
        clamp_mask = (
            (df['Target size'] >= max_target - 0.01) & 
            (np.abs(df['Constant'] - 2.0) < 0.01) &
            (df['Stride Number'] > 50)  # Exclude early baseline
        )
        
        clamp_data = df[clamp_mask].sort_values('Stride Number')
        
        if clamp_data.empty:
            return []
        
        # Group consecutive strides
        clamp_data['gap'] = clamp_data['Stride Number'].diff() > 1
        clamp_data['period_id'] = clamp_data['gap'].cumsum()
        
        clamps = []
        for period_id in clamp_data['period_id'].unique():
            period = clamp_data[clamp_data['period_id'] == period_id]
            if len(period) >= 15:  # Minimum clamp length
                clamps.append({
                    'start': period['Stride Number'].min(),
                    'end': period['Stride Number'].max(),
                    'data': period
                })
        
        return sorted(clamps, key=lambda x: x['start'])
    
    @staticmethod
    def _get_pre_clamp_performance(df: pd.DataFrame, clamp_start: int) -> Optional[float]:
        """Get performance just before clamp period."""
        search_start = max(1, clamp_start - 50)
        search_end = clamp_start - 1
        
        learning_data = df[
            (df['Stride Number'] >= search_start) & 
            (df['Stride Number'] <= search_end) &
            (df['Target size'] < df['Target size'].max() - 0.01)
        ]
        
        if len(learning_data) >= 10:
            return learning_data.tail(15)['Sum of gains and steps'].mean()
        
        return None

# ==============================================================================
# DATASET CREATION
# ==============================================================================

class DatasetBuilder:
    """Build analysis-ready datasets."""
    
    def __init__(self, subjects: Dict[str, Subject], config: Config):
        self.subjects = subjects
        self.config = config
    
    def build_metrics_dataframe(self, 
                               metric_func: callable,
                               metric_name: str) -> pd.DataFrame:
        """Build dataframe for a specific metric across all subjects."""
        rows = []
        
        for subject_id, subject in self.subjects.items():
            row = {
                'subject_id': subject_id,
                'age': subject.age
            }
            
            # Add metadata including tape scores
            metadata_cols = [
                'mot_noise', 
                'pref_asymmetry',
                'min_const_tape_score',
                'pref_const_tape_score', 
                'max_const_tape_score'
            ]
            for key in metadata_cols:
                if key in subject.metadata:
                    row[key] = subject.metadata[key]
            
            # Calculate metric for each trial/condition combination
            for trial in self.config.trial_types:
                df = subject.get_trial(trial)
                if df is not None:
                    # Overall metric for trial
                    row[f'{trial}_{metric_name}'] = metric_func(df)
                    
                    # Metric by condition
                    for condition in self.config.conditions:
                        row[f'{trial}_{metric_name}_{condition}'] = metric_func(df, condition)
            
            rows.append(row)
        
        return pd.DataFrame(rows)
    
    def build_comprehensive_dataset(self) -> pd.DataFrame:
        """Build dataset with all standard metrics."""
        metrics = {
            'sr': MetricsCalculator.calculate_success_rate,
            'msl': MetricsCalculator.calculate_mean_stride_length,
            'sd': MetricsCalculator.calculate_stride_variability,
            'learning': MetricsCalculator.calculate_learning
        }
        
        dfs = []
        for metric_name, metric_func in metrics.items():
            df = self.build_metrics_dataframe(metric_func, metric_name)
            df = df.set_index('subject_id')
            dfs.append(df)
        
        # Merge all metrics, keeping metadata from first df
        result = dfs[0]
        for df in dfs[1:]:
            # Only merge metric columns, not metadata
            metric_cols = [col for col in df.columns if any(
                m in col for m in ['_sr', '_msl', '_sd', '_learning']
            )]
            result = result.join(df[metric_cols])
        
        return result.reset_index()


# ==============================================================================
# AGE-STRATIFIED ANOVA (New Addition)
# ==============================================================================

class AgeStratifiedANOVA:
    """Age-stratified ANOVA analysis."""
    
    def __init__(self, df: pd.DataFrame, config: Config):
        self.df = df
        self.config = config
        
    def run_age_stratified_anova(self, 
                                 dependent_var: str,
                                 age_groups: Dict[str, Tuple[float, float]] = None,
                                 covariates: List[str] = None,
                                 min_subjects_per_group: int = 5) -> Dict:
        """
        Run age-stratified ANOVA for any dependent variable.
        
        Parameters:
        -----------
        dependent_var : str
            Name of the dependent variable (e.g., 'sr', 'msl', 'learning')
        age_groups : dict
            Age group definitions {name: (min_age, max_age)}
        covariates : list
            List of covariates to include in analysis
        min_subjects_per_group : int
            Minimum subjects required per age group
            
        Returns:
        --------
        Dict with ANOVA results for each age group
        """
        
        # Default age groups
        if age_groups is None:
            age_groups = {
                'younger': (7, 12),
                'middle': (12, 15),
                'older': (15, 18)
            }
        
        # Default covariates including tape scores
        if covariates is None:
            covariates = [
                'mot_noise', 
                'pref_asymmetry',
                'min_const_tape_score',
                'pref_const_tape_score',
                'max_const_tape_score'
            ]
        
        results = {
            'dependent_variable': dependent_var,
            'age_groups': age_groups,
            'total_subjects': len(self.df),
            'group_analyses': {},
            'between_group_comparison': None
        }
        
        # Assign age groups
        df = self.df.copy()
        df['age_group'] = self._assign_age_groups(df['age'], age_groups)
        
        # Analyze each age group
        for group_name in age_groups.keys():
            group_df = df[df['age_group'] == group_name]
            
            if len(group_df) < min_subjects_per_group:
                results['group_analyses'][group_name] = {
                    'error': f'Insufficient subjects (n={len(group_df)})'
                }
                continue
            
            # Run within-group analysis
            group_results = self._analyze_group(
                group_df, dependent_var, covariates
            )
            group_results['n_subjects'] = len(group_df)
            group_results['mean_age'] = group_df['age'].mean()
            group_results['age_range'] = (group_df['age'].min(), group_df['age'].max())
            
            results['group_analyses'][group_name] = group_results
        
        # Between-group comparison
        results['between_group_comparison'] = self._compare_between_groups(
            df, dependent_var, age_groups
        )
        
        return results
    
    def _assign_age_groups(self, ages: pd.Series, 
                          age_groups: Dict[str, Tuple[float, float]]) -> pd.Series:
        """Assign age group labels based on age ranges."""
        labels = pd.Series('unassigned', index=ages.index)
        
        for group_name, (min_age, max_age) in age_groups.items():
            mask = (ages >= min_age) & (ages < max_age)
            labels[mask] = group_name
        
        return labels
    
    def _analyze_group(self, group_df: pd.DataFrame, 
                      dependent_var: str,
                      covariates: List[str]) -> Dict:
        """Analyze a single age group."""
        results = {}
        
        # Prepare long format data for ANOVA
        long_data = self._prepare_long_format(group_df, dependent_var)
        
        if long_data.empty:
            return {'error': 'No valid data for analysis'}
        
        # Main effects ANOVA
        if 'trial' in long_data.columns and len(long_data['trial'].unique()) > 1:
            try:
                trial_aov = pg.rm_anova(
                    data=long_data, 
                    dv='value',
                    within='trial', 
                    subject='subject'
                )
                results['trial_effect'] = {
                    'F': float(trial_aov['F'].iloc[0]),
                    'p': float(trial_aov['p-unc'].iloc[0]),
                    'eta2': float(trial_aov['ng2'].iloc[0]),
                    'significant': float(trial_aov['p-unc'].iloc[0]) < 0.05
                }
            except Exception as e:
                results['trial_effect'] = {'error': str(e)}
        
        if 'condition' in long_data.columns and len(long_data['condition'].unique()) > 1:
            try:
                condition_aov = pg.rm_anova(
                    data=long_data,
                    dv='value',
                    within='condition',
                    subject='subject'
                )
                results['condition_effect'] = {
                    'F': float(condition_aov['F'].iloc[0]),
                    'p': float(condition_aov['p-unc'].iloc[0]),
                    'eta2': float(condition_aov['ng2'].iloc[0]),
                    'significant': float(condition_aov['p-unc'].iloc[0]) < 0.05
                }
            except Exception as e:
                results['condition_effect'] = {'error': str(e)}
        
        # Interaction effect if both factors present
        if all(col in long_data.columns for col in ['trial', 'condition']):
            if len(long_data['trial'].unique()) > 1 and len(long_data['condition'].unique()) > 1:
                try:
                    interaction_aov = pg.rm_anova(
                        data=long_data,
                        dv='value',
                        within=['trial', 'condition'],
                        subject='subject'
                    )
                    # Find interaction row
                    interaction_row = interaction_aov[
                        interaction_aov['Source'].str.contains('trial * condition', na=False)
                    ]
                    if not interaction_row.empty:
                        results['interaction'] = {
                            'F': float(interaction_row['F'].iloc[0]),
                            'p': float(interaction_row['p-unc'].iloc[0]),
                            'eta2': float(interaction_row['ng2'].iloc[0]),
                            'significant': float(interaction_row['p-unc'].iloc[0]) < 0.05
                        }
                except Exception as e:
                    results['interaction'] = {'error': str(e)}
        
        # Mean values by trial/condition
        results['means'] = self._calculate_means(long_data)
        
        # Covariate effects
        results['covariate_effects'] = self._analyze_covariates(
            group_df, dependent_var, covariates
        )
        
        # Post-hoc comparisons if significant main effect
        if results.get('trial_effect', {}).get('significant', False):
            results['posthoc'] = self._posthoc_comparisons(long_data, 'trial')
        
        return results
    
    def _prepare_long_format(self, df: pd.DataFrame, 
                            dependent_var: str) -> pd.DataFrame:
        """Convert wide format to long format for ANOVA."""
        long_rows = []
        
        for _, row in df.iterrows():
            subject_id = row.get('subject_id', row.name)
            
            # Find all relevant columns for this dependent variable
            for col in df.columns:
                if dependent_var in col and not col.endswith('_condition'):
                    # Parse trial and condition from column name
                    parts = col.split('_')
                    
                    # Determine structure based on parts
                    trial = None
                    condition = None
                    
                    if len(parts) >= 2:
                        # Could be trial_metric or trial_metric_condition
                        if parts[0] in self.config.trial_types:
                            trial = parts[0]
                            if len(parts) >= 3 and parts[-1] in self.config.conditions:
                                condition = parts[-1]
                    
                    if trial and pd.notna(row[col]):
                        long_row = {
                            'subject': subject_id,
                            'trial': trial,
                            'value': row[col]
                        }
                        if condition:
                            long_row['condition'] = condition
                        
                        long_rows.append(long_row)
        
        return pd.DataFrame(long_rows)
    
    def _calculate_means(self, long_data: pd.DataFrame) -> Dict:
        """Calculate means for each factor level."""
        means = {}
        
        if 'trial' in long_data.columns:
            means['by_trial'] = long_data.groupby('trial')['value'].agg(['mean', 'std', 'count']).to_dict('index')
        
        if 'condition' in long_data.columns:
            means['by_condition'] = long_data.groupby('condition')['value'].agg(['mean', 'std', 'count']).to_dict('index')
        
        if all(col in long_data.columns for col in ['trial', 'condition']):
            means['by_trial_condition'] = long_data.groupby(['trial', 'condition'])['value'].agg(['mean', 'std', 'count']).to_dict('index')
        
        return means
    
    def _analyze_covariates(self, df: pd.DataFrame, 
                           dependent_var: str,
                           covariates: List[str]) -> Dict:
        """Analyze covariate relationships."""
        covariate_results = {}
        
        # Calculate mean dependent variable across trials
        dv_cols = [col for col in df.columns if dependent_var in col]
        if not dv_cols:
            return covariate_results
        
        # Create mean score across all relevant columns
        df['dv_mean'] = df[dv_cols].mean(axis=1)
        
        for covariate in covariates:
            if covariate in df.columns:
                valid_data = df[['dv_mean', covariate]].dropna()
                
                if len(valid_data) > 3:
                    r, p = pearsonr(valid_data['dv_mean'], valid_data[covariate])
                    covariate_results[covariate] = {
                        'correlation': r,
                        'p_value': p,
                        'n': len(valid_data)
                    }
        
        return covariate_results
    
    def _posthoc_comparisons(self, long_data: pd.DataFrame, 
                            factor: str) -> Dict:
        """Run post-hoc pairwise comparisons."""
        posthoc_results = {}
        
        levels = long_data[factor].unique()
        
        for i, level1 in enumerate(levels):
            for level2 in levels[i+1:]:
                data1 = long_data[long_data[factor] == level1]['value']
                data2 = long_data[long_data[factor] == level2]['value']
                
                if len(data1) > 0 and len(data2) > 0:
                    t_stat, p_val = ttest_ind(data1, data2)
                    
                    # Calculate Cohen's d
                    pooled_std = np.sqrt((data1.std()**2 + data2.std()**2) / 2)
                    cohens_d = (data1.mean() - data2.mean()) / pooled_std if pooled_std > 0 else 0
                    
                    comparison_key = f'{level1}_vs_{level2}'
                    posthoc_results[comparison_key] = {
                        't_statistic': t_stat,
                        'p_value': p_val,
                        'cohens_d': cohens_d,
                        'mean_diff': data1.mean() - data2.mean()
                    }
        
        return posthoc_results
    
    def _compare_between_groups(self, df: pd.DataFrame, 
                               dependent_var: str,
                               age_groups: Dict) -> Dict:
        """Compare dependent variable between age groups."""
        # Calculate mean across trials for each subject
        dv_cols = [col for col in df.columns if dependent_var in col]
        if not dv_cols:
            return {'error': 'No columns found for dependent variable'}
        
        df['dv_mean'] = df[dv_cols].mean(axis=1)
        
        # One-way ANOVA between age groups
        group_data = []
        group_names = []
        
        for group_name in age_groups.keys():
            group_df = df[df['age_group'] == group_name]
            data = group_df['dv_mean'].dropna()
            
            if len(data) > 0:
                group_data.append(data.values)
                group_names.append(group_name)
        
        if len(group_data) < 2:
            return {'error': 'Insufficient groups for comparison'}
        
        from scipy.stats import f_oneway
        f_stat, p_value = f_oneway(*group_data)
        
        return {
            'groups': group_names,
            'F': f_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        }

# ==============================================================================
# STATISTICAL ANALYSIS
# ==============================================================================

class StatisticalAnalyzer:
    """Statistical analysis tools."""
    
    def __init__(self, df: pd.DataFrame, config: Config):
        self.df = df
        self.config = config
    
    def anova_trial_effect(self, dependent_var: str) -> Dict:
        """Run repeated measures ANOVA for trial type effect."""
        # Prepare long format data
        long_data = []
        
        for _, row in self.df.iterrows():
            subject_id = row['subject_id']
            
            for trial in self.config.trial_types:
                col = f'{trial}_{dependent_var}'
                if col in row and pd.notna(row[col]):
                    long_data.append({
                        'subject': subject_id,
                        'trial': trial,
                        'value': row[col]
                    })
        
        if not long_data:
            return {'error': 'No data for ANOVA'}
        
        df_long = pd.DataFrame(long_data)
        
        # Run ANOVA
        try:
            aov = pg.rm_anova(data=df_long, dv='value', 
                            within='trial', subject='subject')
            return {
                'F': float(aov['F'].iloc[0]),
                'p': float(aov['p-unc'].iloc[0]),
                'eta2': float(aov['ng2'].iloc[0])
            }
        except Exception as e:
            return {'error': str(e)}
    
    def correlation_with_age(self, dependent_var: str) -> Dict:
        """Calculate correlation between age and a metric."""
        results = {}
        
        for trial in self.config.trial_types:
            col = f'{trial}_{dependent_var}'
            if col in self.df.columns:
                valid = self.df[['age', col]].dropna()
                if len(valid) > 5:
                    r, p = pearsonr(valid['age'], valid[col])
                    results[trial] = {'r': r, 'p': p, 'n': len(valid)}
        
        return results
    
    def age_group_comparison(self, dependent_var: str, 
                           age_groups: Dict[str, Tuple[float, float]]) -> Dict:
        """Compare metric across age groups."""
        # Add age group column
        df = self.df.copy()
        df['age_group'] = 'unknown'
        
        for group_name, (min_age, max_age) in age_groups.items():
            mask = (df['age'] >= min_age) & (df['age'] < max_age)
            df.loc[mask, 'age_group'] = group_name
        
        results = {}
        
        for trial in self.config.trial_types:
            col = f'{trial}_{dependent_var}'
            if col not in df.columns:
                continue
            
            group_means = {}
            group_data = []
            
            for group_name in age_groups.keys():
                group_df = df[df['age_group'] == group_name]
                data = group_df[col].dropna()
                
                if len(data) > 0:
                    group_means[group_name] = {
                        'mean': data.mean(),
                        'std': data.std(),
                        'n': len(data)
                    }
                    group_data.append(data.values)
            
            # Run one-way ANOVA if we have multiple groups
            if len(group_data) > 1:
                from scipy.stats import f_oneway
                f_stat, p_value = f_oneway(*group_data)
                results[trial] = {
                    'groups': group_means,
                    'f_stat': f_stat,
                    'p_value': p_value
                }
            else:
                results[trial] = {'groups': group_means}
        
        return results

# ==============================================================================
# VISUALIZATION
# ==============================================================================

class Visualizer:
    """Create visualizations."""
    
    def __init__(self, config: Config):
        self.config = config
        self.colors = {
            'vis1': '#1f77b4',
            'invis': '#ff7f0e',
            'vis2': '#2ca02c',
            'max': '#d62728',
            'min': '#9467bd'
        }
    
    def plot_age_vs_metric(self, df: pd.DataFrame, metric: str, 
                          save_path: Optional[Path] = None) -> plt.Figure:
        """Plot age vs any metric."""
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        for i, trial in enumerate(self.config.trial_types):
            ax = axes[i]
            col = f'{trial}_{metric}'
            
            if col in df.columns:
                valid = df[['age', col]].dropna()
                
                if not valid.empty:
                    ax.scatter(valid['age'], valid[col], 
                             alpha=0.6, color=self.colors[trial])
                    
                    # Add trendline
                    if len(valid) > 2:
                        z = np.polyfit(valid['age'], valid[col], 1)
                        p = np.poly1d(z)
                        ax.plot(valid['age'], p(valid['age']), 
                               'r--', alpha=0.8)
                        
                        r, pval = pearsonr(valid['age'], valid[col])
                        ax.text(0.05, 0.95, f'r={r:.2f}\np={pval:.3f}',
                               transform=ax.transAxes,
                               bbox=dict(boxstyle='round', facecolor='white'))
            
            ax.set_xlabel('Age (years)')
            ax.set_ylabel(metric.replace('_', ' ').title())
            ax.set_title(trial.upper())
            ax.grid(True, alpha=0.3)
        
        plt.suptitle(f'Age vs {metric.replace("_", " ").title()}')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=self.config.figure_dpi)
        
        return fig
    
    def plot_trial_comparison(self, df: pd.DataFrame, metric: str,
                            save_path: Optional[Path] = None) -> plt.Figure:
        """Compare metric across trial types."""
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        
        data_by_trial = []
        labels = []
        colors_list = []
        
        for trial in self.config.trial_types:
            col = f'{trial}_{metric}'
            if col in df.columns:
                data = df[col].dropna()
                if not data.empty:
                    data_by_trial.append(data)
                    labels.append(trial.upper())
                    colors_list.append(self.colors[trial])
        
        if data_by_trial:
            bp = ax.boxplot(data_by_trial, labels=labels, patch_artist=True)
            
            for patch, color in zip(bp['boxes'], colors_list):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        ax.set_ylabel(metric.replace('_', ' ').title())
        ax.set_title(f'{metric.replace("_", " ").title()} by Trial Type')
        ax.grid(True, alpha=0.3)
        
        if save_path:
            plt.savefig(save_path, dpi=self.config.figure_dpi)
        
        return fig
    
    def plot_condition_comparison(self, df: pd.DataFrame, metric: str, trial: str,
                                 save_path: Optional[Path] = None) -> plt.Figure:
        """Compare metric across conditions for a specific trial."""
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        
        max_col = f'{trial}_{metric}_max'
        min_col = f'{trial}_{metric}_min'
        
        if max_col in df.columns and min_col in df.columns:
            valid = df[[max_col, min_col]].dropna()
            
            if not valid.empty:
                # Scatter plot
                ax.scatter(valid[min_col], valid[max_col], alpha=0.6)
                
                # Unity line
                lims = [
                    np.min([ax.get_xlim(), ax.get_ylim()]),
                    np.max([ax.get_xlim(), ax.get_ylim()])
                ]
                ax.plot(lims, lims, 'k--', alpha=0.5)
                
                # Stats
                from scipy.stats import ttest_rel
                t_stat, p_val = ttest_rel(valid[max_col], valid[min_col])
                
                ax.text(0.05, 0.95, 
                       f'Max > Min: p={p_val:.3f}\nn={len(valid)}',
                       transform=ax.transAxes,
                       bbox=dict(boxstyle='round', facecolor='white'))
        
        ax.set_xlabel(f'Min Target - {metric.replace("_", " ").title()}')
        ax.set_ylabel(f'Max Target - {metric.replace("_", " ").title()}')
        ax.set_title(f'{trial.upper()}: Max vs Min Target')
        ax.grid(True, alpha=0.3)
        
        if save_path:
            plt.savefig(save_path, dpi=self.config.figure_dpi)
        
        return fig

# ==============================================================================
# MAIN ANALYSIS CLASS
# ==============================================================================

class MotorLearningAnalysis:
    """Main analysis coordinator."""
    
    def __init__(self, metadata_path: str, data_root_dir: str, 
                 config: Optional[Config] = None):
        self.config = config or Config()
        self.loader = DataLoader(metadata_path, data_root_dir)
        self.subjects = self.loader.load_all_subjects()
        self.dataset_builder = DatasetBuilder(self.subjects, self.config)
        self.visualizer = Visualizer(self.config)
        
        print(f"Loaded {len(self.subjects)} subjects")
    
    def build_dataset(self, metric_func: callable = None, 
                     metric_name: str = None) -> pd.DataFrame:
        """Build dataset with specified metric or all metrics."""
        if metric_func and metric_name:
            return self.dataset_builder.build_metrics_dataframe(metric_func, metric_name)
        else:
            return self.dataset_builder.build_comprehensive_dataset()
    
    def analyze(self, df: pd.DataFrame, dependent_var: str) -> Dict:
        """Run statistical analysis for a dependent variable."""
        analyzer = StatisticalAnalyzer(df, self.config)
        
        results = {
            'anova': analyzer.anova_trial_effect(dependent_var),
            'age_correlation': analyzer.correlation_with_age(dependent_var),
            'age_groups': analyzer.age_group_comparison(
                dependent_var,
                {'younger': (7, 12), 'middle': (12, 15), 'older': (15, 18)}
            )
        }
        
        return results
    
    def visualize(self, df: pd.DataFrame, dependent_var: str):
        """Create standard visualizations for a dependent variable."""
        # Age vs metric
        fig1 = self.visualizer.plot_age_vs_metric(
            df, dependent_var,
            self.config.figures_dir / f'age_vs_{dependent_var}.png'
        )
        
        # Trial comparison
        fig2 = self.visualizer.plot_trial_comparison(
            df, dependent_var,
            self.config.figures_dir / f'trial_comparison_{dependent_var}.png'
        )
        
        # Condition comparisons for each trial
        for trial in self.config.trial_types:
            fig3 = self.visualizer.plot_condition_comparison(
                df, dependent_var, trial,
                self.config.figures_dir / f'condition_comparison_{trial}_{dependent_var}.png'
            )
        
        plt.close('all')


# ==============================================================================
# ENHANCED VISUALIZATION FOR AGE-STRATIFIED ANOVA
# ==============================================================================

class AgeStratifiedVisualizer:
    """Visualization for age-stratified ANOVA results."""
    
    def __init__(self, config: Config):
        self.config = config
        self.colors = {
            'younger': '#1f77b4',
            'middle': '#ff7f0e', 
            'older': '#2ca02c',
            'significant': '#28a745',
            'non_significant': '#dc3545'
        }
    
    def plot_age_stratified_results(self, 
                                   anova_results: Dict,
                                   save_path: Optional[Path] = None) -> plt.Figure:
        """Create comprehensive visualization of age-stratified ANOVA results."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Extract valid groups
        valid_groups = [name for name, results in anova_results['group_analyses'].items()
                       if 'error' not in results and 'trial_effect' in results]
        
        if not valid_groups:
            return fig
        
        # 1. Effect sizes by age group
        self._plot_effect_sizes(axes[0, 0], anova_results, valid_groups)
        
        # 2. Mean values by trial and age group
        self._plot_means_by_group(axes[0, 1], anova_results, valid_groups)
        
        # 3. Covariate effects
        self._plot_covariate_effects(axes[1, 0], anova_results, valid_groups)
        
        # 4. Summary statistics
        self._plot_summary_table(axes[1, 1], anova_results, valid_groups)
        
        plt.suptitle(f'Age-Stratified ANOVA: {anova_results["dependent_variable"]}',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=self.config.figure_dpi)
        
        return fig
    
    def _plot_effect_sizes(self, ax, results, valid_groups):
        """Plot effect sizes for each age group."""
        effect_sizes = []
        p_values = []
        
        for group in valid_groups:
            group_data = results['group_analyses'][group]
            if 'trial_effect' in group_data:
                effect_sizes.append(group_data['trial_effect']['eta2'])
                p_values.append(group_data['trial_effect']['p'])
        
        colors = [self.colors['significant'] if p < 0.05 else self.colors['non_significant']
                 for p in p_values]
        
        bars = ax.bar(valid_groups, effect_sizes, color=colors, alpha=0.7)
        
        # Add significance markers
        for bar, p_val in zip(bars, p_values):
            height = bar.get_height()
            sig_marker = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
            ax.text(bar.get_x() + bar.get_width()/2, height + 0.01,
                   sig_marker, ha='center', fontweight='bold')
        
        ax.set_ylabel('Effect Size (η²)')
        ax.set_title('Trial Effect by Age Group')
        ax.set_ylim(0, max(effect_sizes) * 1.2 if effect_sizes else 1)
    
    def _plot_means_by_group(self, ax, results, valid_groups):
        """Plot mean values by trial type for each age group."""
        trial_types = ['vis1', 'invis', 'vis2']
        x_pos = np.arange(len(trial_types))
        width = 0.25
        
        for i, group in enumerate(valid_groups):
            group_data = results['group_analyses'][group]
            if 'means' in group_data and 'by_trial' in group_data['means']:
                means = []
                for trial in trial_types:
                    if trial in group_data['means']['by_trial']:
                        means.append(group_data['means']['by_trial'][trial]['mean'])
                    else:
                        means.append(0)
                
                ax.bar(x_pos + i * width, means, width, 
                      label=group, alpha=0.8)
        
        ax.set_xlabel('Trial Type')
        ax.set_ylabel('Mean Value')
        ax.set_title('Mean Values by Trial and Age Group')
        ax.set_xticks(x_pos + width)
        ax.set_xticklabels([t.upper() for t in trial_types])
        ax.legend()
    
    def _plot_covariate_effects(self, ax, results, valid_groups):
        """Plot heatmap of covariate correlations."""
        # Extract covariate data
        covariates = set()
        for group in valid_groups:
            group_data = results['group_analyses'][group]
            if 'covariate_effects' in group_data:
                covariates.update(group_data['covariate_effects'].keys())
        
        if not covariates:
            ax.text(0.5, 0.5, 'No covariate data', ha='center', va='center')
            ax.set_title('Covariate Effects')
            return
        
        covariates = sorted(list(covariates))
        correlation_matrix = np.zeros((len(covariates), len(valid_groups)))
        
        for j, group in enumerate(valid_groups):
            group_data = results['group_analyses'][group]
            if 'covariate_effects' in group_data:
                for i, cov in enumerate(covariates):
                    if cov in group_data['covariate_effects']:
                        correlation_matrix[i, j] = group_data['covariate_effects'][cov]['correlation']
        
        im = ax.imshow(correlation_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
        ax.set_xticks(range(len(valid_groups)))
        ax.set_xticklabels(valid_groups)
        ax.set_yticks(range(len(covariates)))
        ax.set_yticklabels(covariates)
        ax.set_title('Covariate Correlations')
        
        # Add colorbar
        plt.colorbar(im, ax=ax)
    
    def _plot_summary_table(self, ax, results, valid_groups):
        """Create summary statistics table."""
        ax.axis('off')
        
        table_data = []
        headers = ['Age Group', 'N', 'Mean Age', 'Trial p', 'Cond p']
        
        for group in valid_groups:
            group_data = results['group_analyses'][group]
            row = [
                group,
                str(group_data.get('n_subjects', 'N/A')),
                f"{group_data.get('mean_age', 0):.1f}",
                f"{group_data.get('trial_effect', {}).get('p', 1):.3f}",
                f"{group_data.get('condition_effect', {}).get('p', 1):.3f}"
            ]
            table_data.append(row)
        
        table = ax.table(cellText=table_data, colLabels=headers,
                        cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)
        
        ax.set_title('Summary Statistics')

# ==============================================================================
# USAGE EXAMPLE
# ==============================================================================

def main():
    """Example usage."""
    # Initialize analysis
    analysis = MotorLearningAnalysis(
        metadata_path='muh_metadata_tape.csv',
        data_root_dir='muh_data/'
    )
    
    # Build comprehensive dataset
    df = analysis.build_dataset()
    
    # Apply motor noise filter
    if 'mot_noise' in df.columns:
        df = df[df['mot_noise'] <= 0.3]
    
    print(f"Dataset shape: {df.shape}")
    print(f"Age range: {df['age'].min():.1f} - {df['age'].max():.1f}")
    
    # Analyze specific metrics
    metrics_to_analyze = ['sr', 'msl', 'sd', 'learning']
    
    for metric in metrics_to_analyze:
        print(f"\nAnalyzing {metric}...")
        
        # Run analysis
        results = analysis.analyze(df, metric)
        
        # Create visualizations
        analysis.visualize(df, metric)
        
        # Print results
        if 'error' not in results['anova']:
            print(f"  ANOVA: F={results['anova']['F']:.2f}, p={results['anova']['p']:.3f}")
        
        for trial, corr_data in results['age_correlation'].items():
            print(f"  Age correlation ({trial}): r={corr_data['r']:.2f}, p={corr_data['p']:.3f}")
    
    # Custom metric example
    print("\nCalculating retention...")
    retention_df = analysis.dataset_builder.build_metrics_dataframe(
        lambda df, cond=None: MetricsCalculator.calculate_retention(df, 1),
        'retention'
    )
    
    # Save results
    df.to_csv(analysis.config.data_dir / 'processed_metrics.csv', index=False)
    
    print(f"\nAnalysis complete. Results saved to {analysis.config.base_output_dir}")

if __name__ == "__main__":
    main()

Loaded 111 subjects
Dataset shape: (111, 38)
Age range: 7.2 - 17.9

Analyzing sr...
  ANOVA: F=60.82, p=0.000
  Age correlation (vis1): r=0.62, p=0.000
  Age correlation (invis): r=0.53, p=0.000
  Age correlation (vis2): r=0.59, p=0.000

Analyzing msl...
  ANOVA: F=8.26, p=0.000
  Age correlation (vis1): r=0.18, p=0.130
  Age correlation (invis): r=0.21, p=0.054
  Age correlation (vis2): r=0.26, p=0.011

Analyzing sd...
  ANOVA: F=41.90, p=0.000
  Age correlation (vis1): r=-0.42, p=0.000
  Age correlation (invis): r=-0.31, p=0.005
  Age correlation (vis2): r=-0.29, p=0.003

Analyzing learning...

Calculating retention...

Analysis complete. Results saved to motor_learning_output


In [2]:
main()

Loaded 111 subjects
Dataset shape: (111, 38)
Age range: 7.2 - 17.9

Analyzing sr...
  ANOVA: F=60.82, p=0.000
  Age correlation (vis1): r=0.62, p=0.000
  Age correlation (invis): r=0.53, p=0.000
  Age correlation (vis2): r=0.59, p=0.000

Analyzing msl...
  ANOVA: F=8.26, p=0.000
  Age correlation (vis1): r=0.18, p=0.130
  Age correlation (invis): r=0.21, p=0.054
  Age correlation (vis2): r=0.26, p=0.011

Analyzing sd...
  ANOVA: F=41.90, p=0.000
  Age correlation (vis1): r=-0.42, p=0.000
  Age correlation (invis): r=-0.31, p=0.005
  Age correlation (vis2): r=-0.29, p=0.003

Analyzing learning...

Calculating retention...

Analysis complete. Results saved to motor_learning_output
